# 01 — Single Layer Perceptron (SLP)

Este notebook integra e adapta a ideia do repositório **whoisraibolt/Single-Layer-Perceptron** para a trilha deste tutorial.

Fluxo:
1. Definição dos dados 2D linearmente separáveis
2. Implementação do algoritmo do Perceptron de Rosenblatt
3. Treinamento com atualização de pesos e bias
4. Avaliação e visualização da fronteira de decisão


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f'Seed fixa: {SEED}')

## Dados de exemplo (portados do repositório de referência)

Cada amostra possui:
- `x1`, `x2`: coordenadas 2D
- `y`: classe alvo em `{-1, +1}`


In [ ]:
data = np.array([
    [0.72, 0.82, -1], [0.91, -0.69, -1], [0.03, 0.93, -1], [0.12, 0.25, -1], [0.96, 0.47, -1],
    [0.80, -0.75, -1], [0.46, 0.98, -1], [0.66, 0.24, -1], [0.72, -0.15, -1], [0.35, 0.01, -1],
    [-0.11, 0.10, 1], [0.31, -0.96, 1], [0.00, -0.26, 1], [-0.43, -0.65, 1], [0.57, -0.97, 1],
    [-0.72, -0.64, 1], [-0.25, -0.43, 1], [-0.12, -0.90, 1], [-0.58, 0.62, 1], [-0.77, -0.76, 1],
], dtype=np.float32)

X = data[:, :2]
y = data[:, 2].astype(np.int32)

print('X shape:', X.shape)
print('Classes:', np.unique(y))

In [ ]:
def plot_points(X, y, title='Dados 2D'):
    plt.figure(figsize=(6, 6))
    plt.scatter(X[y == -1, 0], X[y == -1, 1], c='royalblue', label='Classe -1')
    plt.scatter(X[y ==  1, 0], X[y ==  1, 1], c='crimson', label='Classe +1')
    plt.xlim(-1.1, 1.1)
    plt.ylim(-1.1, 1.1)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.title(title)
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()

plot_points(X, y, title='Dados originais')

## Implementação do Perceptron

Regra de atualização (quando houver erro):
- `w <- w + lr * y_i * x_i`
- `b <- b + lr * y_i`


In [ ]:
def perceptron_train(X, y, learning_rate=1.0, max_epochs=100):
    w = np.zeros(X.shape[1], dtype=np.float32)
    b = 1.0
    history = []

    for epoch in range(1, max_epochs + 1):
        errors = 0
        for xi, yi in zip(X, y):
            output = np.dot(xi, w) + b
            y_pred = 1 if output > 0 else -1

            if y_pred != yi:
                w = w + learning_rate * yi * xi
                b = b + learning_rate * yi
                errors += 1

        history.append(errors)
        if errors == 0:
            break

    return w, b, history

w, b, history = perceptron_train(X, y, learning_rate=1.0, max_epochs=100)
print('Pesos finais:', w)
print('Bias final  :', b)
print('Épocas      :', len(history))
print('Erros/época :', history)

In [ ]:
def perceptron_predict(X, w, b):
    out = np.dot(X, w) + b
    return np.where(out > 0, 1, -1)

y_hat = perceptron_predict(X, w, b)
acc = (y_hat == y).mean()
print(f'Acurácia no conjunto de treino: {acc:.2%}')

In [ ]:
def plot_decision_boundary(X, y, w, b):
    plt.figure(figsize=(6, 6))
    plt.scatter(X[y == -1, 0], X[y == -1, 1], c='royalblue', label='Classe -1')
    plt.scatter(X[y ==  1, 0], X[y ==  1, 1], c='crimson', label='Classe +1')

    x_vals = np.array([-1.1, 1.1])
    if abs(w[1]) > 1e-12:
        y_vals = -(w[0] * x_vals + b) / w[1]
        plt.plot(x_vals, y_vals, 'g-', lw=2, label='Fronteira aprendida')
    else:
        x0 = -b / w[0]
        plt.axvline(x=x0, color='g', lw=2, label='Fronteira aprendida')

    plt.xlim(-1.1, 1.1)
    plt.ylim(-1.1, 1.1)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.title('SLP — Fronteira de decisão')
    plt.xlabel('x1')
    plt.ylabel('x2')
    plt.show()

plot_decision_boundary(X, y, w, b)

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.plot(range(1, len(history) + 1), history, marker='o')
plt.title('Convergência do Perceptron')
plt.xlabel('Época')
plt.ylabel('Número de erros')
plt.grid(alpha=0.3)
plt.show()

## Conclusão

O SLP aprende uma **fronteira linear** para separar as classes. Esse é o ponto de partida natural antes de evoluir para o MLP e, depois, para a CNN.
